In [1]:
import warnings
from rdkit import RDLogger

# 屏蔽 RDKit 警告
RDLogger.DisableLog('rdApp.*')

# 或屏蔽所有 Python 警告
warnings.filterwarnings("ignore")
# 屏蔽 LightGBM 警告
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")

In [6]:
import torch
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import precision_recall_curve, auc
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
import optuna
from rdkit.Chem import Descriptors, AllChem
from tqdm import tqdm  # 导入tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold





In [2]:
# 函数：将SMILES转换为分子描述符和指纹
def smiles_to_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    # 提取描述符
    descriptors = [
        Descriptors.MolWt(mol),  # 分子量
        Descriptors.MolLogP(mol),  # LogP
        Descriptors.NumHDonors(mol),  # 氢键供体数量
        Descriptors.NumHAcceptors(mol)  # 氢键受体数量
    ]
    # 生成Morgan指纹
    fingerprint = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    fingerprint_array = np.zeros((2048,))
    Chem.DataStructs.ConvertToNumpyArray(fingerprint, fingerprint_array)
    # 合并描述符和指纹
    features = np.concatenate([descriptors, fingerprint_array])
    return features


In [3]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from tqdm import tqdm
import optuna
import numpy as np

def train_evaluate_regression_model_with_optuna(model_name, model_class, param_func, X, y, groups):
    def objective(trial):
        params = param_func(trial)
        model = model_class(**params)

        gkf = GroupKFold(n_splits=10)
        maes = []

        for train_idx, val_idx in tqdm(gkf.split(X, y, groups=groups), total=10, desc=f"Training {model_name}"):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            # ✅ 计算 MAE
            mae = mean_absolute_error(y_val, y_pred)
            maes.append(mae)

        return np.mean(maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)

    print(f'Best parameters for {model_name}: {study.best_params}')
    print(f'Best mean MAE: {study.best_value:.4f}')

In [7]:
# 数据预处理
df = pd.read_excel('../fish_unique.xlsx')
labels = df['mgperL'].values
smiles_list = df['SMILES_Canonical_RDKit'].tolist()
endpoints_a = df['endpoint']
Duration_Values_a = df['Duration_Value'].values
effects_a = df['effect']


In [8]:

features = []
new_labels = []
new_smiles_list = []
endpoints = []
Duration_Values = []
effects =[]


for smiles, label,a,b,c in zip(smiles_list, labels,Duration_Values_a,effects_a,endpoints_a):
    feature = smiles_to_features(smiles)
    if feature is not None:
        features.append(feature)
        new_labels.append(label)
        new_smiles_list.append(smiles)
        Duration_Values.append(a)
        effects.append(b)
        endpoints.append(c)

X = np.array(features)
y = np.array(new_labels)
groups = new_smiles_list  # 可直接用于 GroupKFold




In [9]:
def encode_column(zz):
    zz_series = pd.Series(zz)  # 转换为 Series
    unique_values = zz_series.unique()
    if len(unique_values) > 1:
        encoder = OneHotEncoder(sparse_output=False)
        return encoder.fit_transform(zz_series.values.reshape(-1, 1))
    else:
        return None  # 只有一种类别时忽略

Duration_Values =pd.Series(Duration_Values)


# 编码 effect、endpoint 和 species_group 列
effect_encoded = encode_column(effects)
endpoint_encoded = encode_column(endpoints)
#species_encoded = encode_column(df, 'species_group')

# # 将需要的列拼接成输入 X
X = np.hstack((X, Duration_Values.values.reshape(-1, 1)))

# # 拼接编码后的列（如果存在）
for encoded_feature in [effect_encoded, endpoint_encoded]:
     if encoded_feature is not None:
         X = np.hstack((X, encoded_feature))



y=np.log1p(y)

In [10]:
def xgb_param_func(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),   # L1 正则
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0)  # L2 正则
    }
from xgboost import XGBRegressor

train_evaluate_regression_model_with_optuna(
    "XGBoost",
    XGBRegressor,
    xgb_param_func,
    X, y, groups
)

[I 2025-05-16 07:58:24,538] A new study created in memory with name: no-name-2b128dc5-5647-4a75-af04-91a2246d1584
Training XGBoost: 100%|██████████| 10/10 [05:01<00:00, 30.13s/it]
[I 2025-05-16 08:03:25,895] Trial 0 finished with value: 0.9689812106483355 and parameters: {'n_estimators': 380, 'max_depth': 18, 'learning_rate': 0.02047788883535787, 'subsample': 0.7971537925050566, 'colsample_bytree': 0.8784822321267367, 'reg_alpha': 0.4671983771359205, 'reg_lambda': 0.5183311019711306}. Best is trial 0 with value: 0.9689812106483355.
Training XGBoost: 100%|██████████| 10/10 [03:08<00:00, 18.85s/it]
[I 2025-05-16 08:06:34,451] Trial 1 finished with value: 0.9676955243936499 and parameters: {'n_estimators': 243, 'max_depth': 18, 'learning_rate': 0.034101083773459245, 'subsample': 0.8630538651348083, 'colsample_bytree': 0.6793497645249652, 'reg_alpha': 0.9935238161484673, 'reg_lambda': 0.7345185275275388}. Best is trial 1 with value: 0.9676955243936499.
Training XGBoost: 100%|██████████| 10

Best parameters for XGBoost: {'n_estimators': 568, 'max_depth': 20, 'learning_rate': 0.0548419024359913, 'subsample': 0.6199824702604411, 'colsample_bytree': 0.6669061263322058, 'reg_alpha': 0.928075881586975, 'reg_lambda': 0.9869950065551683}
Best mean MAE: 0.9542


In [11]:
from lightgbm import LGBMRegressor

def lgbm_param_func(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'verbose': -1
    }

print("Training LightGBM (Poisson)...")
train_evaluate_regression_model_with_optuna(
    "LightGBM",
    lambda **params: LGBMRegressor(objective="poisson", **params),  # ✅ 加入 Poisson 目标
    lgbm_param_func,
    X, y, groups
)


[I 2025-05-16 10:04:49,576] A new study created in memory with name: no-name-bcd86a03-a8c2-470f-8258-c6dc4ce87a45


Training LightGBM (Poisson)...


Training LightGBM: 100%|██████████| 10/10 [00:28<00:00,  2.81s/it]
[I 2025-05-16 10:05:17,738] Trial 0 finished with value: 1.0782178193953735 and parameters: {'n_estimators': 179, 'max_depth': 11, 'num_leaves': 146, 'learning_rate': 0.028348788292079707, 'feature_fraction': 0.7723394395921611, 'bagging_fraction': 0.7094283087224402, 'bagging_freq': 4, 'reg_alpha': 0.5597793580510801, 'reg_lambda': 0.8808363156734718}. Best is trial 0 with value: 1.0782178193953735.
Training LightGBM: 100%|██████████| 10/10 [00:13<00:00,  1.40s/it]
[I 2025-05-16 10:05:31,773] Trial 1 finished with value: 0.9966530430279332 and parameters: {'n_estimators': 143, 'max_depth': 10, 'num_leaves': 156, 'learning_rate': 0.20440626376851612, 'feature_fraction': 0.9972311109957965, 'bagging_fraction': 0.80615358245378, 'bagging_freq': 7, 'reg_alpha': 0.0945816450836211, 'reg_lambda': 0.8666157398735094}. Best is trial 1 with value: 0.9966530430279332.
Training LightGBM: 100%|██████████| 10/10 [00:16<00:00,  1.68

Best parameters for LightGBM: {'n_estimators': 353, 'max_depth': 20, 'num_leaves': 170, 'learning_rate': 0.11212245188752953, 'feature_fraction': 0.7673505068979276, 'bagging_fraction': 0.9737113542482807, 'bagging_freq': 4, 'reg_alpha': 0.6115055127735106, 'reg_lambda': 0.4867933887145545}
Best mean MAE: 0.9415


In [13]:
import random

# 固定随机种子
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # for multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)  # 设置固定种子


In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import optuna
import numpy as np


class DNNWithSoftplus(nn.Module):
    def __init__(self, input_dim, hidden_sizes, activation):
        super().__init__()
        act_fn = {
            'relu': nn.ReLU(),
            'logistic': nn.Sigmoid(),
            'tanh': nn.Tanh()
        }[activation]
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev_dim, h), act_fn]
            prev_dim = h
        layers += [nn.Linear(prev_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return F.softplus(self.net(x)).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def train_dnn_with_optuna_pytorch(X, y, groups, device=device):
    def dnn_param_func(trial):
        return {
            'hidden_layer_sizes': trial.suggest_categorical(
                'hidden_layer_sizes', [(50,), (100,), (150,), (100, 50), (150, 100, 50)]
            ),
            'activation': trial.suggest_categorical('activation', ['relu', 'logistic', 'tanh']),
            'alpha': trial.suggest_float('alpha', 1e-5, 1e-2, log=True),
            'learning_rate': trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
            'optimizer': trial.suggest_categorical('solver', ['adam', 'sgd'])
        }

    def objective(trial):
        params = dnn_param_func(trial)
        model = DNNWithSoftplus(
            input_dim=X.shape[1],
            hidden_sizes=params['hidden_layer_sizes'],
            activation=params['activation']
        ).to(device)

        optimizer = {
            'adam': torch.optim.Adam,
            'sgd': torch.optim.SGD
        }[params['optimizer']](model.parameters(), lr=params['learning_rate'], weight_decay=params['alpha'])

        loss_fn = nn.MSELoss()
        gkf = GroupKFold(n_splits=10)
        fold_maes = []

        for train_idx, val_idx in gkf.split(X, y, groups=groups):
            X_train, y_train = X[train_idx], y[train_idx]
            X_val, y_val = X[val_idx], y[val_idx]

            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_val = scaler.transform(X_val)

            train_ds = TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).float())
            train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

            model.train()
            for epoch in range(100):
                for xb, yb in train_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    optimizer.zero_grad()
                    pred = model(xb)
                    loss = loss_fn(pred, yb)
                    loss.backward()
                    optimizer.step()

            model.eval()
            with torch.no_grad():
                val_preds = model(torch.tensor(X_val).float().to(device)).cpu().numpy()
                mae = mean_absolute_error(y_val, val_preds)
                fold_maes.append(mae)

        return np.mean(fold_maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)
    print("\n✅ Best Parameters Found:")
    print(study.best_params)
    print(f"Mean MAE = {study.best_value:.4f}")
    return study.best_params


best_dnn_params = train_dnn_with_optuna_pytorch(X, y, groups)

[I 2025-05-16 11:45:40,433] A new study created in memory with name: no-name-4009ff08-83ce-46f7-9d32-a450af723416
[I 2025-05-16 11:50:06,976] Trial 0 finished with value: 0.6454984101408147 and parameters: {'hidden_layer_sizes': (150,), 'activation': 'relu', 'alpha': 5.00263933725825e-05, 'learning_rate_init': 0.00035799429164715097, 'solver': 'sgd'}. Best is trial 0 with value: 0.6454984101408147.
[I 2025-05-16 11:54:32,996] Trial 1 finished with value: 0.6062075626103911 and parameters: {'hidden_layer_sizes': (150,), 'activation': 'tanh', 'alpha': 0.0020763907704092308, 'learning_rate_init': 0.005912777019445019, 'solver': 'sgd'}. Best is trial 1 with value: 0.6062075626103911.
[I 2025-05-16 11:59:32,163] Trial 2 finished with value: 0.5644910448300758 and parameters: {'hidden_layer_sizes': (150, 100, 50), 'activation': 'relu', 'alpha': 0.0006603180799969622, 'learning_rate_init': 0.0007668915069303349, 'solver': 'sgd'}. Best is trial 2 with value: 0.5644910448300758.
[I 2025-05-16 1


✅ Best Parameters Found:
{'hidden_layer_sizes': (150,), 'activation': 'relu', 'alpha': 7.490484670539513e-05, 'learning_rate_init': 0.009345382511909396, 'solver': 'sgd'}
Mean MAE = 0.5133
